# Nightingale: B1 retrained for R-18, the "asked" channel (EXP-018)

This notebook runs `scripts/train_baselines.py` twice on Colab, where the owner chose to run it
(decision D-7). Both runs train logistic regression and XGBoost (B1) on DDXPlus's train split
**plus one masked copy of every patient**, a shortened history (`src/ml/evidence_masks.py`):

1. **B1 +aug**: the features the B1 models already use.
2. **B1′+aug**: the same, with the encoder's "asked" channel, which tells the model which questions
   were asked, so that an unfinished interview no longer looks like a list of denials (R-18).

Comparing the two separates what the channel does from what the masked copies do. Validate is
scored at **full evidence only**: masked validate patients wait for the team to approve a mask rule
(`docs/proposals/05-amendment-3.md`). Each bundle also records what each model answers to a patient
with no findings, where the original XGBoost said atrial fibrillation with probability 1.000. The
test split is never downloaded.

**Before you start:** the notebook asks for a GPU (T4); XGBoost uses it if it is there. If Colab
offers only a CPU, accept it: the run is just slower.

**Then:** *Runtime → Run all*. It takes about 25–35 minutes, most of it downloading and decoding
the 670 MB `train.csv`. At the end your browser downloads **`nightingale_b1_asked.zip`**.

**Keep this notebook and its outputs private.** Models trained on DDXPlus are never published
(`docs/11` §4). Nothing here needs a password or a token.

In [ ]:
# The commit Claude gave you. "master" works too: run.json records the exact commit either way.
COMMIT = "master"

In [ ]:
import os

if not os.path.exists("/content/nightingale"):
    !git clone -q https://github.com/HarshRohila02/nightingale.git /content/nightingale
%cd /content/nightingale
!git checkout -q {COMMIT}
!git log --oneline -1
# XGBoost 2.1 is the version on the owner's laptop, so the laptop can load the trained model.
!pip install -q "xgboost~=2.1" "scikit-learn~=1.5"

In [ ]:
# DDXPlus from Hugging Face, at the snapshot the laptop uses (docs/11 §4).
# test.csv is never fetched before Phase 4 (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "train.csv", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir="data/raw/ddxplus",
    )
!ls -lh data/raw/ddxplus

In [ ]:
# Decode the vocabulary, then keep the 13 chest-pain conditions of each split (task 1a).
!python scripts/decode_ddxplus.py
!python scripts/build_ddxplus_chestpain.py --split train
!python scripts/build_ddxplus_chestpain.py --split validate

In [ ]:
import shutil

DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("XGBoost runs on", DEVICE)
# B1 +aug: the original features, with one masked copy of every train patient.
!python scripts/train_baselines.py --device {DEVICE} --augment --out models/b1_aug

In [ ]:
# B1′+aug: the same, with the "asked" channel.
!python scripts/train_baselines.py --device {DEVICE} --asked-channel --augment --out models/b1_asked_aug

In [ ]:
# One zip to bring back: both bundles (models, metrics, run.json). No patient rows are in it.
import shutil
from google.colab import files

!mkdir -p /content/b1_asked && cp -r models/b1_aug models/b1_asked_aug /content/b1_asked/
archive = shutil.make_archive("/content/nightingale_b1_asked", "zip", "/content/b1_asked")
!ls -lhR /content/b1_asked {archive}
files.download(archive)